##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Zero-cloud hybrid RAG with SQLite FTS5, gemini-embedding-001, and Gemini

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Zero_Cloud_Hybrid_RAG_SQLite_FTS5_Gemini.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/google-gemini/cookbook/blob/main/examples/Zero_Cloud_Hybrid_RAG_SQLite_FTS5_Gemini.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View on GitHub</a>
  </td>
</table>

<br><br>

## Overview

This recipe demonstrates how you can construct an air-gapped, zero-cloud local hybrid Retrieval-Augmented Generation (RAG) system using standard embedded SQLite, the official `google-genai` SDK, `gemini-embedding-001`, and Gemini.

By combining **sparse lexical search** (via SQLite FTS5 BM25) and **dense semantic search** (via `gemini-embedding-001` cosine similarity) using **Reciprocal Rank Fusion (RRF, $k=60$)**, you achieve high retrieval accuracy without external vector database infrastructure.

## Prerequisites and setup

Install the official `google-genai` SDK and configure your API key from Colab `userdata` or environment variables.

In [ ]:
%pip install -q -U "google-genai>=2.9.0" numpy

In [ ]:
import os
from google import genai

try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    api_key = os.environ.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key) if api_key else None
if client:
    print("GenAI client initialized successfully.")
else:
    print("Running in offline simulation mode (API key not detected).")

## Initialize SQLite hybrid RAG store

Create an embedded SQLite store that manages dense vector embeddings alongside a virtual SQLite FTS5 table with aligned document identifiers.

In [ ]:
import sqlite3
from typing import Any, Dict, List, Tuple
import numpy as np

class SQLiteHybridRAGStore:
    """Embedded zero-cloud hybrid storage with SQLite FTS5 and dense vectors."""

    def __init__(self, db_path: str = ":memory:"):
        self.conn = sqlite3.connect(db_path)
        self._init_tables()

    def _init_tables(self) -> None:
        with self.conn:
            self.conn.execute("""
                CREATE TABLE IF NOT EXISTS document_chunks (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    source_file TEXT NOT NULL,
                    content TEXT NOT NULL,
                    embedding BLOB NOT NULL
                );
            """)
            self.conn.execute("""
                CREATE VIRTUAL TABLE IF NOT EXISTS document_chunks_fts USING fts5(
                    content,
                    source_file UNINDEXED,
                    tokenize='unicode61'
                );
            """)

    def insert_chunk(
        self, source_file: str, content: str, embedding: np.ndarray
    ) -> int:
        norm = np.linalg.norm(embedding)
        if norm > 0:
            normalized_vec = (embedding / norm).astype(np.float32)
        else:
            normalized_vec = embedding.astype(np.float32)
        emb_blob = normalized_vec.tobytes()

        with self.conn:
            cursor = self.conn.cursor()
            cursor.execute(
                "INSERT INTO document_chunks "
                "(source_file, content, embedding) VALUES (?, ?, ?)",
                (source_file, content, emb_blob)
            )
            doc_id = cursor.lastrowid
            cursor.execute(
                "INSERT INTO document_chunks_fts "
                "(rowid, content, source_file) VALUES (?, ?, ?)",
                (doc_id, content, source_file)
            )
            return doc_id

    def search_dense(
        self, query_vec: np.ndarray, top_k: int = 5
    ) -> List[Tuple[int, str, str, float]]:
        norm = np.linalg.norm(query_vec)
        if norm > 0:
            q_norm = (query_vec / norm).astype(np.float32)
        else:
            q_norm = query_vec.astype(np.float32)

        cursor = self.conn.cursor()
        cursor.execute(
            "SELECT id, source_file, content, embedding FROM document_chunks"
        )
        results = []
        for doc_id, src, content, blob in cursor.fetchall():
            doc_vec = np.frombuffer(blob, dtype=np.float32)
            score = float(np.dot(q_norm, doc_vec))
            results.append((doc_id, src, content, score))

        return sorted(results, key=lambda x: x[3], reverse=True)[:top_k]

    def search_sparse_bm25(
        self, query_text: str, top_k: int = 5
    ) -> List[Tuple[int, str, str, float]]:
        tokens = [
            t.replace("'", "").replace('"', "")
            for t in query_text.split() if t.strip()
        ]
        if not tokens:
            return []
        sanitized_query = " OR ".join([f'"{t}"' for t in tokens])

        cursor = self.conn.cursor()
        cursor.execute("""
            SELECT rowid, source_file, content, rank
            FROM document_chunks_fts
            WHERE document_chunks_fts MATCH ?
            ORDER BY rank
            LIMIT ?
        """, (sanitized_query, top_k))

        hits = []
        for doc_id, src, content, rank in cursor.fetchall():
            bm25_score = 1.0 / (1.0 + abs(float(rank)))
            hits.append((doc_id, src, content, bm25_score))
        return hits

    def hybrid_search(
        self, query_text: str, query_vec: np.ndarray,
        top_k: int = 3, rrf_k: int = 60
    ) -> List[Dict[str, Any]]:
        dense_hits = self.search_dense(query_vec, top_k=top_k * 2)
        sparse_hits = self.search_sparse_bm25(query_text, top_k=top_k * 2)
        chunk_map = {}
        fused_scores = {}

        # Fuse dense ranks
        for rank, (doc_id, src, content, _) in enumerate(dense_hits, start=1):
            chunk_map[doc_id] = (src, content, "dense")
            fused_scores[doc_id] = (
                fused_scores.get(doc_id, 0.0) + (1.0 / (rrf_k + rank))
            )

        # Fuse sparse BM25 ranks
        for rank, (doc_id, src, content, _) in enumerate(sparse_hits, start=1):
            if doc_id not in chunk_map:
                chunk_map[doc_id] = (src, content, "bm25")
            else:
                src, content, _ = chunk_map[doc_id]
                chunk_map[doc_id] = (src, content, "hybrid")
            fused_scores[doc_id] = (
                fused_scores.get(doc_id, 0.0) + (1.0 / (rrf_k + rank))
            )

        sorted_doc_ids = sorted(
            fused_scores.keys(), key=lambda d: fused_scores[d], reverse=True
        )[:top_k]
        output = []
        for idx, doc_id in enumerate(sorted_doc_ids, start=1):
            src, content, match_type = chunk_map[doc_id]
            output.append({
                "citation_index": idx,
                "doc_id": doc_id,
                "source_file": src,
                "content": content,
                "rrf_score": round(fused_scores[doc_id], 4),
                "match_type": match_type
            })
        return output

print("SQLiteHybridRAGStore defined successfully with aligned doc_id indexing.")

## Ingest documents with embeddings

Populate your SQLite store with sample enterprise documents and compute dense vector embeddings with `gemini-embedding-001`.

In [ ]:
store = SQLiteHybridRAGStore()

documents = [
    (
        "q3_financial_report.pdf",
        "Core infrastructure engineering Q3 total budget was finalized "
        "at 2,340,000 TL with 15 developers."
    ),
    (
        "architecture_specs.md",
        "Zenith AI leverages local on-device small language models "
        "for zero-cloud edge inference."
    ),
    (
        "hr_policy_2026.docx",
        "Quarterly remote work equipment allowance is strictly capped "
        "at 15,000 TL per developer."
    ),
    (
        "cluster_ops.md",
        "Kubernetes horizontal pod autoscaler scales pods when memory "
        "utilization exceeds 80% for 5 minutes."
    )
]

print("Ingesting corpus into embedded SQLite...")
for src, content in documents:
    if client:
        res = client.models.embed_content(
            model="gemini-embedding-001",
            contents=content
        )
        emb = np.array(res.embeddings[0].values, dtype=np.float32)
    else:
        np.random.seed(abs(hash(content)) % 10000)
        emb = np.random.randn(768).astype(np.float32)

    store.insert_chunk(src, content, emb)

print(f"Successfully ingested {len(documents)} document chunks into SQLite.")

## Execute hybrid search with reciprocal rank fusion

Query the store using both natural language keyword search and semantic vector embeddings, fusing the results with RRF ($k=60$).

In [ ]:
query = "What is the quarterly remote work allowance limit in TL?"
print(f"Query: '{query}'\n")

if client:
    q_res = client.models.embed_content(
        model="gemini-embedding-001",
        contents=query
    )
    query_vec = np.array(q_res.embeddings[0].values, dtype=np.float32)
else:
    np.random.seed(abs(hash(query)) % 10000)
    query_vec = np.random.randn(768).astype(np.float32)

results = store.hybrid_search(query, query_vec, top_k=2)

print("--- Retrieved Citations (SQLite FTS5 + gemini-embedding-001 via RRF k=60) ---")
for r in results:
    print(
        f"[{r['citation_index']}] Source: {r['source_file']} | "
        f"Match: {r['match_type'].upper()} | RRF Score: {r['rrf_score']}"
    )
    print(f"    Content: {r['content']}\n")

## Synthesize grounded answers with Gemini

Provide retrieved context to Gemini with structured system instructions to ensure factual responses grounded with bracketed citation markers.

In [ ]:
MODEL_ID = "gemini-2.5-flash"  # @param ["gemini-3.1-pro-preview", "gemini-3.5-flash-lite", "gemini-2.5-pro", "gemini-2.5-flash"] {"allow-input": true, "isTemplate": true}

context_str = "\n\n".join([
    f"[{r['citation_index']}] (Source: {r['source_file']}) {r['content']}"
    for r in results
])

system_instruction = (
    "You are a precise corporate assistant. Answer the user prompt "
    "strictly based on the provided context.\n"
    "For every factual statement or numerical figure, append the exact "
    "source citation index in brackets ([1], [2]).\n\n"
    f"Context:\n{context_str}"
)

print("--- Grounded Model Response ---\n")
if client:
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=query,
        config=genai.types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.1
        )
    )
    print(response.text)
else:
    print(
        "According to the HR policy documentation [1], the quarterly "
        "remote work equipment allowance is strictly capped at 15,000 TL "
        "per developer."
    )

## What's next

To explore further capabilities with Gemini and Google GenAI SDK:

- Explore the [Google GenAI SDK Python Documentation](https://googleapis.github.io/python-genai/) for additional API parameters and multimodal inputs.
- Check out the [Gemini API Overview](https://ai.google.dev/gemini-api/docs) to learn about model capabilities and context windows.
- Visit the [Google Gemini Cookbook Repository](https://github.com/google-gemini/cookbook) for more recipes on structured outputs, function calling, and RAG architectures.